# Victory Status in Chess

## Introduction

### Background Information

Chess is a game that humans have been playing for over 1400 years. Today, the game is often played virtually through applications like chess.com or lichess.org. Due to the popularity of these sites, thousands of games may be played each day and the amount of data collected from all games is massive and extensive.

The Elo system is a ranking system to assign a numerical value to the chess skill of any given player. The original purpose of this system is to be able to determine the odds that a player will win over their opponent. Upon winning, a player’s Elo rating goes up, and the opposite is true upon losing. The amount these values change depends on the discrepancy between the two ratings.

There are a few ways for a chess match to end. A player could checkmate their opponent (win), the match could end in a stalemate (draw), the time clock could run out on the opponents clock (win), or they could resign (win). 


### Predictive Question

In this project, we will develop a model to predict the victory status of any given chess match before it starts. The 4 possible predictions are:
1. Mate
2. Draw
3. Resign
4. Out of time

We will use 4 variables to feed into the prediction model:
1. White lower rating? (Boolean)
2. Difference in rating (+Integer)
3. Average rating (Double)
4. Ranked? (Boolean)

All variables serve a purpose to help predict the outcome, but there are different reasonings for each variable. Variable (1) includes the advantage (or disadvantage) in starting the game (because white plays first) as it pertains to rating. Variable (2) includes the difference in rating and variable (3) will be used to guage the magnitude both players are playing at (avg rating). Variable (4) includes whether or not the game is ranked (e.x. maybe the players will resign less in an ranked game?).

### Dataset

To make this predictive model, we will use a data set available on kaggle.com:
https://www.kaggle.com/datasets/datasnaek/chess

This dataset is a collection of data from over 20,000 chess matches. It includes many variables such as player I.D., moves, openings, and time stamps for the creation and ending of matches. The only variables we are interested in for this model is:
1. White rating (white_rating)
2. Black rating (black_rating)
3. Victory status (victory_status)
4. Rated (rated)

## Methods & Results

### Overview & Description

#### Reading

To access the data in a consistent and permanent way, we download the dataset from kaggle.com, upload it to a github repo, and read the csv from a link to that repo.

#### Wrangling

To wrangle the data we select only the variables we need (listed above). We then mutate the dataset to include 3 more variables, and then select for them:
1. White lower rating (wlr) is created using a simple boolean operator between the black and white ratings.
2. Difference in rating (dir) is created using simple subtraction.
3. Average rating (avg_rating) is created by adding white and black rating and dividing by two.

There are two boolean variables that also need to be mutated, but will have the same name: rated and wlr. This is because our classification engine only accepts numeric values, so we will convert TRUE values to 1 and FALSE values to 0.

In addition to those three variables we will also have in this mutated dataset, we will keep the (rated) and (victory_status) variables.

The final dataset will include 5 variables:
1. wlr (is white lower in rating)
2. dir (difference in rating)
3. avg_rating (average rating of both players)
4. rated (whether or not the game is rated)
5. victory_status (how the game ended)

#### Splitting

As this is clearly a classification problem, we use the K-NN Classification algorithm to train our classifier. We use the inital_split function to randomly split our chess dataset into a 75% training dataset and 25% testing dataset.

#### Choosing K

Instead of choosing an arbitrary value of K, we use cross validation to find the best value of K. We create 5 splits using the vfold_cv function. We train our classifier by putting tune() as the value for the parameter neighbors, as we are trying to find an optimised value. As we finish the cross validation, we find that the K value with the highest accuracy is 41, and therefore that is our best K value.

#### Training Model

We re-train our classifier, this time using K = 41. After re-training the classifier, we use the predict function and test our classifier's results against the testing data.

#### Predicting New Data

The overall accuracy of our tuned classifier with the testing data comes out to be 54.86%. Though this may not seem like much, it is a considerable improvement over 25%, which would be the default prediction accuracy (as there are 4 classes). Chess games are also dependent on several abstract factors like player strategy which could not be used as data for the classifier.

### Reading & Wrangling

In [30]:
# Setting seed for reproducible randomisation
set.seed(2022)

# Run this first
library(repr)
library(tidyverse)
library(tidymodels)

In [31]:
chess_URL <- "https://raw.githubusercontent.com/m1ndst0rmz/DSCI-GroupProject/main/games.csv"
chess_raw <- read_csv(chess_URL)

Parsed with column specification:
cols(
  id = col_character(),
  rated = col_logical(),
  created_at = col_double(),
  last_move_at = col_double(),
  turns = col_double(),
  victory_status = col_character(),
  winner = col_character(),
  increment_code = col_character(),
  white_id = col_character(),
  white_rating = col_double(),
  black_id = col_character(),
  black_rating = col_double(),
  moves = col_character(),
  opening_eco = col_character(),
  opening_name = col_character(),
  opening_ply = col_double()
)



In [39]:
chess <- chess_raw %>%
mutate(wlr = white_rating < black_rating) %>%
mutate(dir = abs(white_rating - black_rating)) %>%
mutate(avg_rating = (white_rating + black_rating)/2) %>% 
mutate(victory_status = as_factor(victory_status))

# Converts Boolean to 1 or 0d
chess$wlr <- chess$wlr * 1
chess$rated <- chess$rated * 1
chess <- select(chess, rated, wlr, dir, avg_rating, victory_status)

glimpse(chess)

Rows: 20,058
Columns: 5
$ rated          <dbl> 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, …
$ wlr            <dbl> 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, …
$ dir            <dbl> 309, 61, 4, 15, 54, 248, 97, 695, 47, 172, 109, 486, 5…
$ avg_rating     <dbl> 1345.5, 1291.5, 1498.0, 1446.5, 1496.0, 1126.0, 1471.5…
$ victory_status <fct> outoftime, resign, mate, mate, mate, draw, resign, res…


### Preliminary Data Visualizations

PUT STUFF HERE

### Data Analysis

In [33]:
# Cross validation and choosing best K to retrain model


# Splitting into training and testing
chess_split <- initial_split(data = chess, prop = 0.75, strata = victory_status)
chess_train <- training(chess_split)
chess_test <- testing(chess_split)

# Creating 5 splits of training data for cross validation
chess_vfold <- vfold_cv(data = chess_train, v = 5, strata = victory_status)

# Creating the recipe
chess_recipe <- recipe(victory_status ~ ., data = chess_train) %>% 
                step_scale(dir, avg_rating) %>% 
                step_center(dir, avg_rating)    # Not scaling and centering the boolean variables

# Tuning the classifier and finding the best K using cross validation
tune_spec <-    nearest_neighbor(weight_func = "rectangular", neighbors = tune()) %>% 
                set_engine("kknn") %>%
                set_mode("classification")

k_vals <-       tibble(neighbors = seq(from = 1, to = 50, by = 2))

knn_results <-  workflow() %>% 
                add_recipe(chess_recipe) %>% 
                add_model(tune_spec) %>% 
                tune_grid(resamples = chess_vfold, grid = k_vals) %>% 
                collect_metrics() %>% 
                filter(.metric == "accuracy")

knn_results


neighbors,.metric,.estimator,mean,n,std_err,.config
<dbl>,<chr>,<chr>,<dbl>,<int>,<dbl>,<chr>
1,accuracy,multiclass,0.4750070,5,0.0032884712,Model01
3,accuracy,multiclass,0.4914253,5,0.0019908763,Model02
5,accuracy,multiclass,0.5055835,5,0.0039844046,Model03
7,accuracy,multiclass,0.5177482,5,0.0033759166,Model04
9,accuracy,multiclass,0.5244616,5,0.0023014997,Model05
11,accuracy,multiclass,0.5287823,5,0.0022521358,Model06
13,accuracy,multiclass,0.5344328,5,0.0027684666,Model07
15,accuracy,multiclass,0.5382214,5,0.0017260450,Model08
17,accuracy,multiclass,0.5410134,5,0.0026025789,Model09


In [37]:
# Choosing best K based on highest accuracy
k_acc <-    knn_results %>%
            arrange(desc(mean)) %>%
            slice(1) %>% 
            pull(neighbors)

k_acc

[1] 41

In [38]:
# Re-training model with best K
knn_spec <- nearest_neighbor(weight_func = "rectangular", neighbors = k_acc) %>% 
            set_engine("kknn") %>% 
            set_mode("classification")

# Fitting re-trained model to training set
knn_fit <-  workflow() %>% 
            add_recipe(chess_recipe) %>% 
            add_model(knn_spec) %>%
            fit(data = chess_train)


# Predictions with testing set
chess_predictions <-    predict(knn_fit, chess_test) %>% 
                        bind_cols(chess_test) %>% 
                        metrics(truth = victory_status, estimate = .pred_class) %>% 
                        filter(.metric == "accuracy")

chess_predictions

.metric,.estimator,.estimate
<chr>,<chr>,<dbl>
accuracy,multiclass,0.5486637


Therefore, accuracy against testing set is 54.86%.

### Analysis Visualization

PUT STUFF HERE

## Discussion

### Summarize What You Found

From our tuned classifier with the testing data, we have found that the overall acccuracy is 54.86%, the default prediction accuracy we produced for we only used 4 classes, It was, of course, still quite surprising how low the accuracy model was. Keep in mind that there are serveral abstract factors in chess that could not be represented as data for the classifier.

### Discuss Whether This Is What You Expected to Find

Of course, we originally thought the accuracy number would be much higher than 54.86%. Maybe somewhere in the 70-80% range. Maybe this was due to the fact that throughout this course we have been working with higher accuracies, but our final number does make sense given the problem domain.

### Dsicuss What Impact Could Such Findings Have

The unpredictable nature of chess due to evolving human strategy, a high branching factor, and environmental factors that are not part of the game have led to our accuracy being lower than ideal. The one specific factor that makes a higher accuracy so challenging the diverse range of human strategy. Although this factor is able to make big impacts to the accuracy, this data is very hard to represent as data. A players propensity to take risks, play defensively, or sacrifice pieces is very hard to quantify. 

### Discuss What Futuree Questions Could This Lead To

Due to the fact that we only processed a subset of the data due to server and computational limitations. Our next step is to aim towards increasing our computational efficiency to be able to process larger datasets.
We have been discussing very early on about how we could use Elo to predict other factors such as predicting the chances of winning a game based on past elo ratings.